In [1]:
import pandas as pd
import numpy as np
import pycountry
from tqdm import tqdm

In [2]:
# PyCountry mapping
records = []
for c in pycountry.countries:
    records.append({
        "country_name": c.name,
        "iso3":         c.alpha_3,
        "iso2":         c.alpha_2,
        "iso_numeric":  c.numeric,
    })

df_iso = pd.DataFrame(records)
print(df_iso.shape)
print(df_iso.head())

(249, 4)
    country_name iso3 iso2 iso_numeric
0          Aruba  ABW   AW         533
1    Afghanistan  AFG   AF         004
2         Angola  AGO   AO         024
3       Anguilla  AIA   AI         660
4  Åland Islands  ALA   AX         248


In [24]:
# Index of economic freedom
df_eco_free = pd.read_csv('../Clean/heritage-index-of-economic-freedom.csv', skiprows=4)
# Join to get iso3 codes
df_eco_free = df_eco_free.merge(df_iso, left_on="Country", right_on="country_name", how="left")
# Limit to columns we need
df_eco_free = df_eco_free[["iso3", "Index Year", "Overall Score", "Investment Freedom", "Financial Freedom", "Tax Burden"]]
df_eco_free = df_eco_free.drop_duplicates(subset=["iso3", "Index Year"], keep="last")
df_eco_free

,iso3,Index Year,Overall Score,Investment Freedom,Financial Freedom,Tax Burden
0,ALB,2026,68.0,60.0,60.0,89.3
1,ALB,2025,66.6,60.0,60.0,88.8
2,ALB,2024,64.8,70.0,70.0,88.8
3,ALB,2023,65.3,70.0,70.0,89.1
4,ALB,2022,66.6,70.0,70.0,89.1
...,...,...,...,...,...,...
1450,GBR,1999,76.2,70.0,90.0,62.3
1451,GBR,1998,76.5,70.0,90.0,60.8
1452,GBR,1997,76.4,70.0,90.0,61.7
1453,GBR,1996,76.4,70.0,90.0,62.1


In [25]:
# Gravity (Lang + Dist)
df_gravity = pd.read_csv('../Clean/cepii_gravity_language.csv')
# Join to get iso3 codes for both origin and destination
# It is iso3, though we renamed to iso in data.
df_gravity = df_gravity.merge(df_iso.add_suffix('_o'), left_on="iso3_o", right_on="iso3_o", how="left")
df_gravity = df_gravity.merge(df_iso.add_suffix('_d'), left_on="iso3_d", right_on="iso3_d", how="left")
# Select only some columns
df_gravity = df_gravity[["iso3_o", "iso3_d", "year", "distance_km", "shared_border", "common_legal_origin", "language_proximity", "language_spoken_share", "common_official_language"]]
# Remove duplicates
df_gravity = df_gravity.drop_duplicates(subset=["iso3_o", "iso3_d", "year"], keep="last")
df_gravity

,iso3_o,iso3_d,year,distance_km,shared_border,common_legal_origin,language_proximity,language_spoken_share,common_official_language
0,ABW,ABW,1948,NaN,NaN,NaN,NaN,NaN,NaN
1,ABW,ABW,1949,NaN,NaN,NaN,NaN,NaN,NaN
2,ABW,ABW,1950,NaN,NaN,NaN,NaN,NaN,NaN
3,ABW,ABW,1951,NaN,NaN,NaN,NaN,NaN,NaN
4,ABW,ABW,1952,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
4699291,ZWE,ZWE,2017,43.0,0.0,1.0,NaN,NaN,NaN
4699292,ZWE,ZWE,2018,43.0,0.0,1.0,NaN,NaN,NaN
4699293,ZWE,ZWE,2019,43.0,0.0,1.0,NaN,NaN,NaN
4699294,ZWE,ZWE,2020,42.0,0.0,1.0,NaN,NaN,NaN


In [26]:
# IMF CPIS
df_cpis = pd.read_csv('../Clean/IMF_CPIS.csv')
# join by iso3 codes
df_cpis = df_cpis.merge(df_iso.add_suffix("_o"), left_on="REF_AREA_LABEL", right_on="country_name_o", how="left")
# join COMP_BREAKDOWN_1_LABEL to to get iso3 to get destination country
df_cpis = df_cpis.merge(df_iso.add_suffix("_d"), left_on="COMP_BREAKDOWN_1_LABEL", right_on="country_name_d", how="left")
# Now we only keep the necessary columns
df_cpis = df_cpis[["iso3_o", "iso3_d", "year", "value"]]
# rename value to cpis
df_cpis = df_cpis.rename({"value": "cpis"}, axis=1)
# remove duplicates
df_cpis = df_cpis.drop_duplicates(subset=["iso3_o", "iso3_d", "year"], keep="last")
# Add the lagged cpis as well
df_cpis = df_cpis.sort_values(by=["iso3_o", "iso3_d", "year"])  # Ensure it's sorted before shifting
df_cpis["cpis_t-1"] = df_cpis.groupby(["iso3_o", "iso3_d"])["cpis"].shift(1)
df_cpis

,iso3_o,iso3_d,year,cpis,cpis_t-1
33870,ABW,AGO,2001,0.000000e+00,NaN
70971,ABW,AGO,2002,0.000000e+00,0.0
107218,ABW,AGO,2003,0.000000e+00,0.0
145424,ABW,AGO,2004,0.000000e+00,0.0
187656,ABW,AGO,2005,0.000000e+00,0.0
...,...,...,...,...,...
2264143,NaN,NaN,2019,5.416588e+07,NaN
2566925,NaN,NaN,2020,2.382153e+07,NaN
2876087,NaN,NaN,2021,1.581715e+07,NaN
3173904,NaN,NaN,2022,1.941212e+07,NaN


In [27]:
# MSCI Indices
df_msci = pd.read_csv('../Clean/MSCI_Country_ETFs_Yearly.csv')
# Change to get pct returns instead of price
df_msci = df_msci.sort_values(by=["iso3", "Year"])  # Ensure it's sorted before calculating returns
df_msci["Return"] = df_msci.groupby("iso3")["Price"].pct_change()
# Limit columns
df_msci = df_msci[["iso3", "Year", "Return"]]
# Rename Return to MSCI_Return for clarity
df_msci = df_msci.rename({"Return": "MSCI_Return"}, axis=1)
df_msci

/var/folders/_x/74827dzs033cwdj2j4gch59r0000gn/T/ipykernel_59197/956951936.py:5: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_msci["Return"] = df_msci.groupby("iso3")["Price"].pct_change()


,iso3,Year,MSCI_Return
1488,ARE,1996,NaN
1489,ARE,1997,NaN
1490,ARE,1998,NaN
1491,ARE,1999,NaN
1492,ARE,2000,NaN
...,...,...,...
1142,ZAF,2022,-0.051778
1143,ZAF,2023,0.015083
1144,ZAF,2024,0.071604
1145,ZAF,2025,0.752008


In [28]:
# PWT
df_pwt = pd.read_csv('../Clean/pwt.csv')
df_pwt = df_pwt.rename({"countrycode": "iso3",}, axis=1)
df_pwt

,Real GDP,Capital Stock,Worker Population,Total Population,Labour Share,Depreciation Rate,TFP,Interest Rate,Investment Share,Human Capital,iso3,year
0,368.250916,776.961548,NaN,0.058950,NaN,0.040318,NaN,NaN,0.431354,NaN,ABW,1970
1,401.719330,857.151245,NaN,0.058781,NaN,0.039650,NaN,NaN,0.418163,NaN,ABW,1971
2,438.229523,944.586182,NaN,0.058895,NaN,0.039095,NaN,NaN,0.419161,NaN,ABW,1972
3,478.057892,1039.993408,NaN,0.059435,NaN,0.038641,NaN,NaN,0.403070,NaN,ABW,1973
4,521.506104,1143.999634,NaN,0.060122,NaN,0.038273,NaN,NaN,0.390737,NaN,ABW,1974
...,...,...,...,...,...,...,...,...,...,...,...,...
11196,44445.031250,107682.148438,5.272343,15.271368,0.533381,0.059923,1.057740,0.275977,0.130147,2.713408,ZWE,2019
11197,40970.785156,109248.289062,5.206007,15.526888,0.533381,0.060208,0.973187,0.263760,0.119450,2.746586,ZWE,2020
11198,44440.199219,110979.773438,5.298346,15.797210,0.533381,0.060136,1.000000,0.205160,0.153113,2.770661,ZWE,2021
11199,47147.550781,114134.882812,5.344455,16.069056,0.533381,0.059788,0.997026,0.263871,0.172489,2.795292,ZWE,2022


## Full dataframe

In [29]:
### FULL ###
# Get min date for each country in the gravity data
min_year = max(df_gravity["year"].min(), df_cpis["year"].min(), df_eco_free["Index Year"].min(), df_msci["Year"].min())
min_year

1997

In [30]:
# Limit all datasets to this min year
df_gravity = df_gravity[df_gravity["year"] >= min_year]
df_cpis = df_cpis[df_cpis["year"] >= min_year]
df_eco_free = df_eco_free[df_eco_free["Index Year"] >= min_year]
df_msci = df_msci[df_msci["Year"] >= min_year]

In [31]:
# Join to one dataframe
# gravity + destination eco freedom
df_full = df_gravity.merge(df_eco_free.add_suffix("_d"), left_on=["iso3_d", "year"], right_on=["iso3_d", "Index Year_d"], how="left")
# full + cpis
df_full = df_full.merge(df_cpis, left_on=["iso3_o", "iso3_d", "year"], right_on=["iso3_o", "iso3_d", "year"], how="left")
# full + msci d
df_full = df_full.merge(df_msci.add_suffix("_d"), left_on=["iso3_d", "year"], right_on=["iso3_d", "Year_d"], how="left")
# full + msci o
df_full = df_full.merge(df_msci.add_suffix("_o"), left_on=["iso3_o", "year"], right_on=["iso3_o", "Year_o"], how="left")
# full + pwt o
df_full = df_full.merge(df_pwt.add_suffix("_o"), left_on=["iso3_o", "year"], right_on=["iso3_o", "year_o"], how="left")
# full + pwt d
df_full = df_full.merge(df_pwt.add_suffix("_d"), left_on=["iso3_d", "year"], right_on=["iso3_d", "year_d"], how="left")
# Limit to some columns
df_full

,iso3_o,iso3_d,year,distance_km,shared_border,common_legal_origin,language_proximity,language_spoken_share,common_official_language,Index Year_d,...,Capital Stock_d,Worker Population_d,Total Population_d,Labour Share_d,Depreciation Rate_d,TFP_d,Interest Rate_d,Investment Share_d,Human Capital_d,year_d
0,ABW,ABW,1997,32.0,0.0,1.0,NaN,NaN,NaN,NaN,...,9336.132812,0.037406,0.081106,0.634907,0.035945,NaN,0.102071,0.345264,NaN,1997.0
1,ABW,ABW,1998,32.0,0.0,1.0,NaN,NaN,NaN,NaN,...,10134.475586,0.038947,0.083777,0.638232,0.035930,NaN,0.104848,0.340526,NaN,1998.0
2,ABW,ABW,1999,32.0,0.0,1.0,NaN,NaN,NaN,NaN,...,10909.001953,0.040518,0.086472,0.655211,0.035447,NaN,0.087118,0.312591,NaN,1999.0
3,ABW,ABW,2000,32.0,0.0,1.0,NaN,NaN,NaN,NaN,...,11437.366211,0.041918,0.088761,0.645106,0.034938,NaN,0.085381,0.309222,NaN,2000.0
4,ABW,ABW,2001,32.0,0.0,1.0,NaN,NaN,NaN,NaN,...,11965.093750,0.042579,0.090305,0.645106,0.034852,NaN,0.089992,0.297306,NaN,2001.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476220,ZWE,ZWE,2017,43.0,0.0,1.0,NaN,NaN,NaN,NaN,...,102027.710938,5.204640,14.812482,0.533381,0.057745,1.132940,0.269749,0.119595,2.648248,2017.0
1476221,ZWE,ZWE,2018,43.0,0.0,1.0,NaN,NaN,NaN,NaN,...,105167.187500,5.235959,15.034452,0.533381,0.059023,1.153515,0.284309,0.133667,2.680630,2018.0
1476222,ZWE,ZWE,2019,43.0,0.0,1.0,NaN,NaN,NaN,NaN,...,107682.148438,5.272343,15.271368,0.533381,0.059923,1.057740,0.275977,0.130147,2.713408,2019.0
1476223,ZWE,ZWE,2020,42.0,0.0,1.0,NaN,NaN,NaN,NaN,...,109248.289062,5.206007,15.526888,0.533381,0.060208,0.973187,0.263760,0.119450,2.746586,2020.0


In [32]:
df_full.isna().sum() / len(df_full)

iso3_o                      0.000000
iso3_d                      0.000000
year                        0.000000
distance_km                 0.078968
shared_border               0.078968
common_legal_origin         0.083863
language_proximity          0.365916
language_spoken_share       0.365916
common_official_language    0.365916
Index Year_d                0.827160
Overall Score_d             0.835556
Investment Freedom_d        0.834239
Financial Freedom_d         0.834074
Tax Burden_d                0.835556
cpis                        0.898944
cpis_t-1                    0.906732
Year_d                      0.794239
MSCI_Return_d               0.869630
Year_o                      0.794239
MSCI_Return_o               0.869630
Real GDP_o                  0.243128
Capital Stock_o             0.259259
Worker Population_o         0.261399
Total Population_o          0.243128
Labour Share_o              0.426008
Depreciation Rate_o         0.259259
TFP_o                       0.506173
I

In [33]:
# Get last year nan values
last_year = df_full["year"].max()
df_full[df_full["year"] == last_year].isna().sum() / len(df_full[df_full["year"] == last_year])

iso3_o                      0.000000
iso3_d                      0.000000
year                        0.000000
distance_km                 0.064760
shared_border               0.064760
common_legal_origin         0.069654
language_proximity          0.365916
language_spoken_share       0.365916
common_official_language    0.365916
Index Year_d                0.827160
Overall Score_d             0.831276
Investment Freedom_d        0.827160
Financial Freedom_d         0.827160
Tax Burden_d                0.831276
cpis                        0.859964
cpis_t-1                    0.860201
Year_d                      0.794239
MSCI_Return_d               0.794239
Year_o                      0.794239
MSCI_Return_o               0.794239
Real GDP_o                  0.238683
Capital Stock_o             0.259259
Worker Population_o         0.263374
Total Population_o          0.238683
Labour Share_o              0.423868
Depreciation Rate_o         0.259259
TFP_o                       0.506173
I

In [34]:
# Rename the columns to macroeconomic variables
COLUMN_MAP = {
    # ── Identifiers ───────────────────────────────────────────
    "iso3_o"                   : "iso3_i",        # source country ISO3
    "iso3_d"                   : "iso3_j",        # destination country ISO3
    "year"                     : "year",

    # ── Gravity / bilateral variables ─────────────────────────
    "distance_km"              : "dist",          # log this before use
    "shared_border"            : "border",
    "common_legal_origin"      : "legal_i",
    "language_proximity"       : "lang",
    "language_spoken_share"    : "lang_share",
    "common_official_language" : "lang_official",

    # ── Legal / institutional (destination) ───────────────────
    "Overall Score_o"          : "legal_i",       # primary legal index
    "Investment Freedom_o"     : "inv_freedom_i",
    "Financial Freedom_o"      : "fin_freedom_i",
    "Tax Burden_o"             : "tax_i",

    # ── CPIS bilateral flows ───────────────────────────────────
    "cpis"                     : "cpis",          # log this before use
    "cpis_t-1"                 : "cpis_lag1",     # log this before use

    # ── MSCI returns ──────────────────────────────────────────
    "MSCI_Return_o"            : "r_i",
    "MSCI_Return_d"            : "r_j",

    # ── Macro — source country (i) ────────────────────────────
    "Real GDP_o"               : "Y_i",
    "Capital Stock_o"          : "K_i",
    "Worker Population_o"      : "L_i",
    "Total Population_o"       : "pop_i",
    "Labour Share_o"           : "lab_sh_i",      # alpha_i = 1 - lab_sh_i
    "Depreciation Rate_o"      : "delta_i",
    "TFP_o"                    : "A_i",
    "Interest Rate_o"          : "rf_i",
    "Investment Share_o"       : "inv_share_i",
    "Human Capital_o"          : "hc_i",

    # ── Macro — destination country (j) ───────────────────────
    "Real GDP_d"               : "Y_j",
    "Capital Stock_d"          : "K_j",
    "Worker Population_d"      : "L_j",
    "Total Population_d"       : "pop_j",
    "Labour Share_d"           : "lab_sh_j",      # alpha_j = 1 - lab_sh_j
    "Depreciation Rate_d"      : "delta_j",
    "TFP_d"                    : "A_j",
    "Interest Rate_d"          : "rf_j",
    "Investment Share_d"       : "inv_share_j",
    "Human Capital_d"          : "hc_j"
}

df_full = df_full.rename(columns=COLUMN_MAP)

In [35]:
df_full["ln_dist"] = np.log(df_full["dist"])
df_full["ln_cpis"] = np.log(df_full["cpis"])
df_full["ln_cpis_lag1"] = np.log(df_full["cpis_lag1"])

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encounter

In [36]:
remove_cols = ["Index Year_d", "year_d", "year_o", "Year_o", "Year_d"]
df_full = df_full.drop(columns=remove_cols, errors="ignore")

In [37]:
# Fill informational columns that should have a constant values
# Infer them by getting mean std of all columns grouped by
# iso3_i and iso3_j.
grp = df_full.groupby(["iso3_i", "iso3_j"]).agg(["mean", "std", "count"]).reset_index()
display(grp.head())
# Get the columns where mean std is 0, and count > 1 (i.e. there are duplicates with same value)
constant_cols = []
for col in df_full.columns:
    if col in ["iso3_i", "iso3_j", "year"]:
        continue
    mean_std = grp[(col, "std")]
    mean_count = grp[(col, "count")]
    if np.nanmean(mean_std) == 0 and np.nanmean(mean_count) > 1:
        constant_cols.append(col)
print("Constant columns:", constant_cols)

iso3_i iso3_j    year                     dist                 border       \
                   mean       std count     mean       std count   mean  std   
0    ABW    ABW  2009.0  7.359801    25     32.0  0.000000    25    0.0  0.0   
1    ABW    AFG  2009.0  7.359801    25  13192.6  7.868714    25    0.0  0.0   
2    ABW    AGO  2009.0  7.359801    25   9578.6  6.075909    25    0.0  0.0   
3    ABW    AIA  2009.0  7.359801    25    978.0  0.000000    25    0.0  0.0   
4    ABW    ALB  2009.0  7.359801    25   9090.0  0.000000    25    0.0  0.0   

   ...  hc_j   ln_dist                 ln_cpis           ln_cpis_lag1      \
   ... count      mean       std count    mean std count         mean std   
0  ...     0  3.465736  0.000000    25     NaN NaN     0          NaN NaN   
1  ...     0  9.487411  0.000597    25     NaN NaN     0          NaN NaN   
2  ...    25  9.167287  0.000634    25    -inf NaN     7         -inf NaN   
3  ...     0  6.885510  0.000000    25    -inf NaN     7         -inf NaN   
4  ...    25  9.114930  0.000000    25    -inf NaN     8         -inf NaN   

         
  count  
0     0  
1     0  
2     6  
3     6  
4     7  

[5 rows x 116 columns]

Constant columns: ['border', 'legal_i', 'lang', 'lang_share', 'lang_official']


In [38]:
df_full[constant_cols].isna().sum()

border           116575
legal_i          123800
lang             540175
lang_share       540175
lang_official    540175
dtype: int64

In [39]:
# Only select the mean values
grp_mean = df_full.groupby(["iso3_i", "iso3_j"])[constant_cols].mean().reset_index()
# Join to df_full, overwrite the existing constant columns
for col in constant_cols:
    del df_full[col]  # Remove the original column with NaNs

df_full = pd.merge(df_full, grp_mean, on=["iso3_i", "iso3_j"], how="left")
df_full[constant_cols].isna().sum()

border            72250
legal_i           79475
lang             540175
lang_share       540175
lang_official    540175
dtype: int64

In [40]:
# Fill in the bilateral variables for intra-national pairs (where iso3_i == iso3_j)
bil_cols = ["dist", "border", "lang", "lang_share", "lang_official"]

In [41]:
# Get duplicate column names, inspect if they have the same values


In [42]:
# Save to csv
savepath = "../Clean/Final.csv"
df_full.to_csv(savepath, index=False)

In [43]:
df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1476225 entries, 0 to 1476224
Data columns (total 40 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   iso3_i                1476225 non-null  object 
 1   iso3_j                1476225 non-null  object 
 2   year                  1476225 non-null  int64  
 3   dist                  1359650 non-null  float64
 4   Overall Score_d       242757 non-null   float64
 5   Investment Freedom_d  244701 non-null   float64
 6   Financial Freedom_d   244944 non-null   float64
 7   Tax Burden_d          242757 non-null   float64
 8   cpis                  149182 non-null   float64
 9   cpis_lag1             137684 non-null   float64
 10  r_j                   192456 non-null   float64
 11  r_i                   192456 non-null   float64
 12  Y_i                   1117314 non-null  float64
 13  K_i                   1093500 non-null  float64
 14  L_i                   1090341 non-